In [4]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 63.8 MB/s eta 0:00:00


In [5]:
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
court_sentences = [
    ["the", "court", "delivered", "judgment", "in", "favor", "of", "the", "plaintiff"],
    ["the", "defendant", "filed", "an", "appeal", "against", "the", "ruling"],
    ["the", "judge", "presided", "over", "the", "case", "in", "the", "high", "court"],
    ["the", "lawyer", "presented", "evidence", "to", "support", "the", "claim"],
    ["the", "plaintiff", "sought", "compensation", "for", "damages"],
    ["the", "defendant", "denied", "the", "allegations", "in", "court"],
    ["the", "magistrate", "issued", "a", "ruling", "on", "the", "bail", "application"],
    ["the", "court", "of", "appeal", "reviewed", "the", "judgment"],
    ["the", "witness", "gave", "testimony", "before", "the", "court"],
    ["the", "prosecution", "presented", "evidence", "against", "the", "accused"],
]

In [6]:
model = Word2Vec(court_sentences, vector_size=50, window=3, min_count=1)
print("Model trained on", len(court_sentences), "court sentences")

Model trained on 10 court sentences


In [7]:
text_sentences = [" ".join(s) for s in court_sentences]

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(text_sentences)
feature_names = vectorizer.get_feature_names_out()

print("TF-IDF vocabulary size:", len(feature_names))

TF-IDF vocabulary size: 33


In [8]:
def top_keywords(doc_index, top_n=2):
    """Get the most important word(s) in a sentence using TF-IDF"""
    row = tfidf_matrix[doc_index].toarray()[0]
    top_indices = row.argsort()[-top_n:][::-1]
    return [(feature_names[i], round(row[i], 3)) for i in top_indices if row[i] > 0]

print("LEGAL TERM EXPLORER — TF-IDF + WORD2VEC")
print("=" * 55)

for i, sentence in enumerate(text_sentences):
    keywords = top_keywords(i)
    print(f"\nSentence {i+1}: {sentence}")
    print("Top legal keywords (TF-IDF):")
    for word, score in keywords:
        similar = model.wv.most_similar(word, topn=2) if word in model.wv else []
        sim_str = ", ".join([f"{w}({s:.2f})" for w, s in similar])
        print(f"   {word} (tfidf={score})  -> similar in meaning: {sim_str}")

LEGAL TERM EXPLORER — TF-IDF + WORD2VEC

Sentence 1: the court delivered judgment in favor of the plaintiff
Top legal keywords (TF-IDF):
   delivered (tfidf=0.513)  -> similar in meaning: plaintiff(0.25), the(0.25)
   favor (tfidf=0.513)  -> similar in meaning: plaintiff(0.24), presided(0.24)

Sentence 2: the defendant filed an appeal against the ruling
Top legal keywords (TF-IDF):
   filed (tfidf=0.562)  -> similar in meaning: before(0.25), presided(0.20)
   defendant (tfidf=0.478)  -> similar in meaning: denied(0.32), of(0.24)

Sentence 3: the judge presided over the case in the high court
Top legal keywords (TF-IDF):
   presided (tfidf=0.479)  -> similar in meaning: before(0.43), judgment(0.29)
   judge (tfidf=0.479)  -> similar in meaning: support(0.27), compensation(0.24)

Sentence 4: the lawyer presented evidence to support the claim
Top legal keywords (TF-IDF):
   support (tfidf=0.474)  -> similar in meaning: testimony(0.32), judge(0.27)
   claim (tfidf=0.474)  -> similar in mea